# Problem Set 4

### Helper Functions

In [1]:
# Error Function
def error_function(prev, curr):
    error = sum([abs(curr[key] - prev[key]) for key in curr])
    return error / len(curr)

In [2]:
# Gibbs Sampler
import numpy as np

def gibbs(A, s, t, w, burnin, its):
    """Gibbs Sampling algorithm

    Arguments:
        A {n x n matrix} -- adjacency matrix of graph G
        s {integer} -- vertex in set A
        t {integer} -- vertex in set B
        w {n x n matrix} -- weights for each edge
        burnin {integer} -- number of burn-in samples
        its {integer} -- number of iterations after burn-in

    Returns:
        A vector of marginals whose ith component is equal to the probability
        that i ∈ V is in the same part of the cut as s.
    """

    n = A.shape[0]
    steps = burnin + its

    # Start with a valid sample
    sample = np.zeros(n)
    sample[s] = 1
    samples = np.empty((its, n))

    # We'll add any samples after burnin to totals
    totals = np.zeros(n)

    vertices = np.arange(n)
    # Don't consider changing s or t
    vertices = vertices[vertices != s]
    vertices = vertices[vertices != t]


    neighbors = [[idx for idx, edge in enumerate(A[i]) if edge == 1] for i in range(n)]

    for step in range(0, steps):
        # Randomize the order every time
        np.random.shuffle(vertices)

        for i in vertices:
            prob0 = 0
            prob1 = 0

            # Probability of x_i increases by w[i][j] whenever i != j
            for j in neighbors[i]:
                if sample[j] == 1:
                    prob0 += w[i][j]
                else:
                    prob1 += w[i][j]

            # Normalize
            prob1 = prob1 / (prob0 + prob1)

            # Stochastically update sample[i]
            # based on the marginal of i
            # that we calculated
            sample[i] = int(np.random.rand() < prob1)

        if step >= burnin:
            totals += sample
            samples[step - burnin] = sample

    return totals / its, samples

In [3]:
# Generate Permutations
from itertools import product

def generate_permutations(sample):
    indices = [i for i, v in enumerate(sample) if v is None]
    for perm in product([0,1], repeat=len(indices)):
        if perm != ():
            result = sample.copy()
            for idx, val in zip(indices, perm):
                result[idx] = val
            yield result
        else:
            yield sample

In [4]:
# Sum Prod
from collections import defaultdict
from copy import deepcopy
import numpy as np

def sumprod(A, w, fixed, its=3):
    n = A.shape[0]
    s = 0
    t = n - 1
    edges = [(i, j) for i in range(n) for j in range(n) if A[i,j] == 1]

    # These are the only assignments allowed for each variable
    assignments = [[0,1] if x_i == None else [x_i] for x_i in fixed]

    phi = lambda i, x_i: x_i if i == s else (1 - x_i if i == t else 1)
    psi = lambda i, j, x_i, x_j: (w[(i, j)] if (i, j) in w else w[(j, i)]) ** int(x_i != x_j)

    # n x n matrix
    # Stores messages from factor nodes (C) to vertices (i).
    # C_i_messages[C, i] represents the message from the factor C
    # to the vertex i
    C_i_messages = defaultdict(lambda: 1)
    new_C_i_messages = defaultdict(lambda: 1)

    # n x n matrix
    # Stores messages from vertices (i) to factor nodes (C).
    # i_C_messages[i, C] represents the message from the vertex i
    # to the factor C
    i_C_messages = defaultdict(lambda: 1)

    for it in range(its):
        # Messages from C to i
        for (C, i) in edges:
            for x_i in assignments[i]:
                C_i_sum = 0

                # Sum over assignments to x_C
                for x_C in assignments[C]:
                    # Product over k in C \ i = {C}
                    # Would be a loop if I was using cliques and not just edges
                    C_i_sum += psi(C, i, x_C, x_i) * i_C_messages[C, (C, i), x_C]

                new_C_i_messages[(C, i), i, x_i] = C_i_sum

        # Messages from i to C
        for (i, C) in edges:
            for x_i in assignments[i]:
                prod = phi(i, x_i)

                # Product over messages into i
                for (C_prime, k) in edges:
                    # Exclude C -> i
                    if k == i and C_prime != C:
                        prod *= C_i_messages[(C_prime, k), k, x_i]

                i_C_messages[i, (i, C), x_i] = prod

        C_i_messages = deepcopy(new_C_i_messages)

        # Scale
        max_val = max(map(abs, [*C_i_messages.values(), *i_C_messages.values()]))
        if max_val > 0:
            for i in C_i_messages:
                C_i_messages[i] = max(1e-50, C_i_messages[i] / max_val)
            for i in i_C_messages:
                i_C_messages[i] = max(1e-50, i_C_messages[i] / max_val)

    # Calculate beliefs for all i
    beliefs = defaultdict(int)
    for i in range(n):
        for x_i in assignments[i]:
            beliefs[i, x_i] = phi(i, x_i)

            for (C, k) in edges:
                if k == i:
                    beliefs[i, x_i] *= C_i_messages[(C, k), k, x_i]

        total = beliefs[i, 0] + beliefs[i, 1]
        for x_i in assignments[i]:
            beliefs[i, x_i] /= total

    # Calculate beliefs for all C
    C_beliefs = defaultdict(int)
    for (i, j) in edges:
        if (j, i) in C_beliefs:
            continue

        total = 0

        for x_i in assignments[i]:
            for x_j in assignments[j]:
                prob = psi(i, j, x_i, x_j)
                # Message i -> C (x_i)
                prob *= i_C_messages[i, (i, j), x_i]
                # Message j -> C (x_j)
                prob *= i_C_messages[j, (j, i), x_j]

                C_beliefs[(i, j), (x_i, x_j)] = prob
                total += prob

        # Normalize
        for x_i in assignments[i]:
            for x_j in assignments[j]:
                C_beliefs[(i, j), (x_i, x_j)] /= total

    return beliefs, C_beliefs

## Problem 1: Expectation Maximization for Cuts

In [5]:
# Compute ESS for Cuts
from collections import defaultdict
from copy import deepcopy
import math

def compute_ess_for_cuts(A, edges, theta, D, L):
    remember_beliefs = defaultdict(lambda: defaultdict(int))
    M = defaultdict(lambda: 0)

    for m in range(len(D)):
        # Sample at m
        o_m = D[m]

        # Perform inference on o_m
        if None in o_m:
            if tuple(o_m) in remember_beliefs:
                C_beliefs = remember_beliefs[tuple(o_m)]
            else:
                _, C_beliefs = sumprod(A, theta, o_m)
                remember_beliefs[tuple(o_m)] = C_beliefs

        for (i, j) in edges:
            if o_m[i] == None or o_m[j] == None:
                for [x_i, x_j] in generate_permutations([o_m[i], o_m[j]]):
                    if x_i != x_j:
                        M[(i, j)] += C_beliefs[(i, j), (x_i, x_j)]
            else:
                M[(i, j)] += (o_m[i] != o_m[j])

    return M

In [6]:
# Cuts EM
def cutsem(A, L, samples, its=1000):
    """Expectation Maximization for Cuts

    Arguments:
        A {n x n matrix} -- adjacency matrix of graph G
        L {n-dimensional vector} -- latent variables
        samples {n x m matrix} --
            A list of m samples from which we'll
            estimate the weights of the edges in G

    Returns:
        The vector of weights w corresponding to the MLE
        parameters from each edge from the EM algorithm
    """

    n, m = samples.shape

    D = np.array(samples, dtype=object)
    # Set latent variables to None
    D[np.where(L == 1)] = None
    D = np.transpose(D)

    edges = []
    for i in range(n):
        for j in range(n):
            if A[i,j] == 1 and (j,i) not in edges:
                edges.append((i,j))

    # Initial parameters
    theta = defaultdict(lambda: 1)

    for _ in range(its):
        theta_prev = deepcopy(theta)

        # E-step
        # Expected sufficient statistics
        M = compute_ess_for_cuts(A, edges, theta_prev, D, L)

        # Calculate all P(i,j) so we can use them
        # to calculate the Expected Value of each edge
        _, C_beliefs = sumprod(A, theta_prev, [None] * n)

        # M-step
        for (i, j) in edges:
            # (Sum of expected value of sample (i,j) / m) - (Expected value of (i,j) given current parameters)
            theta[(i, j)] += ((M[(i, j)] / m) - (C_beliefs[(i, j), (1, 0)] + C_beliefs[(i, j), (0, 1)]))

        if error_function(theta_prev, theta) < 1e-3:
            break

    return theta

In [7]:
from pprint import pprint
#              a, b, c, d
A = np.array([[0, 1, 1, 0],  # a
              [1, 0, 0, 1],  # b
              [1, 0, 0, 1],  # c
              [0, 1, 1, 0]]) # b

#              a, b, c, d
w = np.array([[0, 5, 1, 0],  # a
              [5, 0, 0, 1],  # b
              [1, 0, 0, 1],  # c
              [0, 1, 1, 0]]) # d

s = 0
t = 3
its = 2**6
burnin = 2**12

samples = gibbs(A, s, t, w, its, burnin)[1]
L = np.array([0,0,0,0])
weights = cutsem(A, L, np.transpose(samples))

print("Approximate Weights:")
pprint(weights)

Approximate Weights:
defaultdict(<function cutsem.<locals>.<lambda> at 0x79f7a91dbbe0>,
            {(0, 1): 1.6710500717163086,
             (0, 2): 1.0068225860595703,
             (1, 3): 0.3289499282836914,
             (2, 3): 0.9931774139404297})


In [8]:
# Case 3: s - [3 cycle] - t

A = np.array([[0,1,0,0,0],[1,0,1,1,0],[0,1,0,1,0],[0,1,1,0,1],[0,0,0,1,0]])
w = np.array([[0,2,0,0,0],[2,0,2,4,0],[0,2,0,3,0],[0,4,3,0,3],[0,0,0,3,0]])

s = 0
t = 4
its = 2**6
burnin = 2**12

samples = gibbs(A, s, t, w, its, burnin)[1]
L = np.array([0,0,1,0,0])
weights = cutsem(A, L, np.transpose(samples))

print("Approximate Weights:")
pprint(weights)

Approximate Weights:
defaultdict(<function cutsem.<locals>.<lambda> at 0x79f7c0100e50>,
            {(0, 1): 2.7975502020109193,
             (1, 2): 1.0,
             (1, 3): 2.049404952858001,
             (2, 3): 1.0,
             (3, 4): 2.9160367728567116})


In [9]:
from pprint import pprint
import math
#              a, b, c
A = np.array([[0, 1, 0],
              [1, 0, 1],
              [0, 1, 0]])

#              a, b, c, d
w = np.array([[0, math.exp(1), 0],
              [math.exp(1), 0, math.exp(1)],
              [0, math.exp(1), 0]])

s = 0
t = 2
its = 2**6
burnin = 2**12

samples = gibbs(A, s, t, w, its, burnin)[1]
L = np.array([0,0,0])
weights = cutsem(A, L, np.transpose(samples))

print("Approximate Weights:")
pprint(weights)

Approximate Weights:
defaultdict(<function cutsem.<locals>.<lambda> at 0x79f7c3b7bb50>,
            {(0, 1): 1.0260162353515625,
             (1, 2): 0.9739837646484375})


In [10]:
# Case 11
from pprint import pprint

A = np.array([
    [0,1,0,0,1],
    [1,0,1,0,0],
    [0,1,0,1,0],
    [0,0,1,0,1],
    [1,0,0,1,0]
])

w = np.array([
    [0,2,0,0,4],
    [2,0,3,0,0],
    [0,3,0,5,0],
    [0,0,5,0,6],
    [4,0,0,6,0]
])

s = 0
t = 4
its = 2**6
burnin = 2**12

samples = gibbs(A, s, t, w, its, burnin)[1]
L = np.array([0,0,1,0,0])
weights = cutsem(A, L, np.transpose(samples))

print("Approximate Weights:")
pprint(weights)

Approximate Weights:
defaultdict(<function cutsem.<locals>.<lambda> at 0x79f7a91fc4c0>,
            {(0, 1): 1.3378995433773415,
             (0, 4): 1.0,
             (1, 2): 1.0,
             (2, 3): 1.0,
             (3, 4): 6.198529656826173})


## Problem 2: EM for Bayesian Networks
### 1. Fit a Bayesian Network

In [11]:
# Load data
import pandas as pd
import numpy as np

voting_data = pd.read_csv('house-votes-84.data', header=None)
voting_data = voting_data.iloc[:, :3]
voting_data[voting_data == '?'] = None
voting_data[voting_data == 'republican'] = 0
voting_data[voting_data == 'democrat'] = 1
voting_data[voting_data == 'n'] = 0
voting_data[voting_data == 'y'] = 1

train = np.array(voting_data.iloc[:300, :3])
test = np.array(voting_data.iloc[300:, :3])


In [12]:
# Compute ESS
from collections import defaultdict
from copy import deepcopy

def compute_ess(G, theta, D):
    n = len(G)

    M = defaultdict(lambda: 0)

    # Collect probabilities from all instances
    for m in range(len(D)):
        # Sample at m
        o_m = D[m]

        # Run inference on 〈G, θ〉 using evidence o[m]
        for i in range(n):
            # Parents
            u = G[i]

            missing_indices = np.where(o_m == None)[0]

            #   P(x_i, x_u | o_m, theta)
            # = Q(x_i, x_u)
            # = Sum of Q(x_?, x_i, x_u) over Val(x_?, x_i, x_u)
            for instantiation in generate_permutations(o_m):
                Q = 1
                for j in missing_indices:
                    x_j = instantiation[j]
                    v = G[j]
                    v_i = instantiation[v]
                    Q *= theta[j, x_j, tuple(v_i)]

                x_i = instantiation[i]
                u_i = tuple(instantiation[u])

                M[i, x_i, u_i] += Q
    return M

In [28]:
# Bayesian EM
def bayesian_em(G, samples):
    n = len(G)

    # Initial parameters
    theta = defaultdict(lambda: 0.5)

    while True:
        theta_prev = deepcopy(theta)

        # E-step
        # Expected sufficient statistics
        M = compute_ess(G, theta, samples)

        # M-step
        for i in range(n):
            for x_i in range(2):
                for u_i in list(product([0,1], repeat=len(G[i]))):
                    theta[i, x_i, u_i] = M[i, x_i, u_i] / (M[i, 0, u_i] + M[i, 1, u_i])

        if error_function(theta_prev, theta) < 1e-20:
            break
    return theta

In [22]:
# Evaluate
def evaluate(G, theta, data):
    correct = 0
    wrong = 0

    for x in data:
        label = x[0]
        features = x[1:]

        P_0 = 0
        P_1 = 0
        # Probability of the votes
        # Partially observable
        if None in features:
            # Add 1 here so the indices are correct after we add the first column back in
            missing_indices = np.where(features == None)[0] + 1

            for partial_instantiation in generate_permutations(features):
                instantiation_0 = np.append([0], partial_instantiation)
                instantiation_1 = np.append([1], partial_instantiation)

                u_i_label = instantiation_0[G[0]]

                prob0 = theta[0, 0, tuple(u_i_label)]
                for i in missing_indices:
                    x_i = instantiation_0[i]
                    u = G[i]
                    u_i = instantiation_0[u]
                    prob0 *= theta[i, x_i, tuple(u_i)]
                P_0 += prob0

                prob1 = theta[0, 1, tuple(u_i_label)]
                for i in missing_indices:
                    x_i = instantiation_1[i]
                    u = G[i]
                    u_i = instantiation_1[u]
                    prob1 *= theta[i, x_i, tuple(u_i)]
                P_1 += prob1
        # Fully observable
        else:
            # Probability of each party
            u = G[0]
            u_i = x[u]

            P_0 = theta[0, 0, tuple(u_i)]
            P_1 = theta[0, 1, tuple(u_i)]

        predicted = int(P_0 < P_1)

        if predicted == label:
            correct += 1
        else:
            wrong += 1

    return correct / (correct + wrong)

### 2. Do different runs of the EM algorithm produce different models?
The EM algorithm implementation I used is deterministic. Given the same data and DAG, the same model will be produced. Another implementation of the EM algorithm might randomize the initial theta parameters. Assuming the function to be maximized has more than one local maximum, each run may converge on different parameters for theta. A random EM algorithm may still converge on the same parameters across multiple runs in the case that there is a single local maximum or the initial theta parameters share the same local maximum.

In [33]:
from pprint import pprint

possible_DAGs = [
    { 0: [], 1: [0], 2: [0,1] },
    { 0: [], 1: [0,2], 2: [0] },
    { 0: [1], 1: [], 2: [0,1] },
    { 0: [1,2], 1: [], 2: [1] },
    { 0: [2], 1: [0,2], 2: [] },
    { 0: [1,2], 1: [2], 2: [] },
]

for i, dag in enumerate(possible_DAGs):
    theta = bayesian_em(dag, train)
    print(f"Model {i + 1}")
    print('  |  '.join(f"{dag[vertex]} -> {vertex}" for vertex in dag))
    pprint(theta)
    print()


Model 1
[] -> 0  |  [0] -> 1  |  [0, 1] -> 2
defaultdict(<function bayesian_em.<locals>.<lambda> at 0x79f760d26050>,
            {(0, 0, ()): 0.3766666666666666,
             (0, 1, ()): 0.6233333333333334,
             (1, 0, (0,)): 0.8125,
             (1, 0, (1,)): 0.39444444444444443,
             (1, 1, (0,)): 0.1875,
             (1, 1, (1,)): 0.6055555555555555,
             (2, 0, (0, 0)): 0.4487179487179488,
             (2, 0, (0, 1)): 0.47058823529411764,
             (2, 0, (1, 0)): 0.5583397255603852,
             (2, 0, (1, 1)): 0.541632225712239,
             (2, 1, (0, 0)): 0.5512820512820512,
             (2, 1, (0, 1)): 0.5294117647058824,
             (2, 1, (1, 0)): 0.44166027443961475,
             (2, 1, (1, 1)): 0.458367774287761})

Model 2
[] -> 0  |  [0, 2] -> 1  |  [0] -> 2
defaultdict(<function bayesian_em.<locals>.<lambda> at 0x79f7a91fc700>,
            {(0, 0, ()): 0.37666666666666665,
             (0, 1, ()): 0.6233333333333333,
             (1, 0, (0, 0)

### 3. Evaluation
The predictions are highly dependent on the model that was fit. The models where party affiliation is dependent on the first vote (models 3, 4, and 6) perform significantly better than the other models.

In models 1 and 2, party affiliation is independent of votes 1 and 2. Therefore, these models always predict the most frequent party from the training data, regardless of any votes. Democrats are the most frequent party in the training data from house-votes-84. The testing data has 80 Democrats and 55 Republicans, thus predicting Democrat every time results in the accuracies of 0.5926 shown by models 1 and 2 below.

Why do models 3, 4, and 6 perform better than model 5? Likely because all of them have an edge from the first vote into party affiliation. It seems that the first vote is a good predictor of party affiliation, whereas the model where party affiliation depends solely on the second vote performs exactly the same as the models where party affiliation is independent.

In [35]:
# Evaluation
for i, dag in enumerate(possible_DAGs):
    theta = bayesian_em(dag, train)
    print(f"Model {i + 1}")
    print('  |  '.join(f"{dag[vertex]} -> {vertex}" for vertex in dag))
    print(f"Accuracy: {evaluate(dag, theta, test)}")
    print()


Model 1
[] -> 0  |  [0] -> 1  |  [0, 1] -> 2
Accuracy: 0.5925925925925926

Model 2
[] -> 0  |  [0, 2] -> 1  |  [0] -> 2
Accuracy: 0.5925925925925926

Model 3
[1] -> 0  |  [] -> 1  |  [0, 1] -> 2
Accuracy: 0.6814814814814815

Model 4
[1, 2] -> 0  |  [] -> 1  |  [1] -> 2
Accuracy: 0.6814814814814815

Model 5
[2] -> 0  |  [0, 2] -> 1  |  [] -> 2
Accuracy: 0.5925925925925926

Model 6
[1, 2] -> 0  |  [2] -> 1  |  [] -> 2
Accuracy: 0.6814814814814815

